# Alytes-ReID: Toad Identification Tool

Upload a photo of a toad and find its match in the database.

**Prerequisites**: Run `01_setup_and_training.ipynb` first to train models and save them to Google Drive.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danort92/Alytes-ReID/blob/main/notebooks/02_toad_reid.ipynb)

## 1. Setup (run once per session)

In [ ]:
# Install and setup
!git clone https://github.com/danort92/Alytes-ReID.git 2>/dev/null; cd Alytes-ReID && git pull
%cd Alytes-ReID
!pip install -r requirements.txt -q

# Mount Google Drive and load models
from google.colab import drive
drive.mount('/content/drive')

DRIVE_MODEL_DIR = '/content/drive/MyDrive/Alytes-ReID/models'

import shutil
from pathlib import Path

local_models = Path('data/models')
local_models.mkdir(parents=True, exist_ok=True)

# Copy models from Drive to local
# shutil.copy2(f'{DRIVE_MODEL_DIR}/detection_best.pt', 'data/models/detection_best.pt')
# shutil.copy2(f'{DRIVE_MODEL_DIR}/reid_model.pt', 'data/models/reid_model.pt')

print('Setup complete!')

## 2. Load Models

In [ ]:
# from src.detection.predict import load_detector
# from src.segmentation.segment import ToadSegmenter
# from src.reid.model import build_model
# from src.reid.database import EmbeddingDatabase
# from src.utils.io import load_yaml
# import torch
#
# # Load configs
# det_config = load_yaml(Path('config/detection.yaml'))
# pre_config = load_yaml(Path('config/preprocessing.yaml'))
# reid_config = load_yaml(Path('config/reid.yaml'))
#
# # Load models
# detector = load_detector(Path('data/models/detection_best.pt'))
# segmenter = ToadSegmenter()
#
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# reid_model = build_model(reid_config).to(device)
# reid_model.load_state_dict(torch.load('data/models/reid_model.pt', map_location=device))
# reid_model.eval()
#
# # Load embedding database
# db = EmbeddingDatabase(
#     embedding_dim=reid_config['model']['embedding_dim'],
#     index_type=reid_config['database']['index_type'],
# )
# db_path = Path('data/models/reid')
# if (db_path / 'index.faiss').exists():
#     db.load(db_path)
#
# print(f'Models loaded. Device: {device}')
# print(f'Database: {db.size} embeddings, {len(db.get_individuals())} individuals')

## 3. Upload & Identify a Toad

In [ ]:
from google.colab import files
import cv2
import matplotlib.pyplot as plt

# Upload image
print('Upload a toad photo:')
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    print(f'Uploaded: {filename}')
    
    # Display uploaded image
    img = cv2.cvtColor(cv2.imread(filename), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.title('Uploaded Image')
    plt.axis('off')
    plt.show()

In [ ]:
# Run full pipeline: detection → segmentation → preprocessing → matching

# from src.detection.predict import detect_toads, get_best_detection
# from src.preprocessing.pipeline import preprocess_toad
# from src.reid.match import match_toad
# from src.utils.visualization import draw_detections, draw_mask_overlay
#
# # Step 1: Detect toad
# detections = detect_toads(detector, Path(filename))
# best_det = get_best_detection(detections)
#
# if best_det is None:
#     print('No toad detected in image!')
# else:
#     print(f'Toad detected (confidence: {best_det["confidence"]:.2f})')
#
#     # Step 2: Segment
#     cropped, mask = segmenter.segment_and_crop(img, best_det['bbox'])
#
#     # Step 3: Preprocess
#     standardized = preprocess_toad(cropped, mask, pre_config)
#
#     # Step 4: Match
#     result = match_toad(
#         reid_model, db, standardized,
#         top_k=reid_config['inference']['top_k'],
#         threshold=reid_config['inference']['similarity_threshold'],
#         device=device,
#     )
#
#     # Display results
#     fig, axes = plt.subplots(1, 3, figsize=(15, 5))
#     axes[0].imshow(draw_detections(img, detections))
#     axes[0].set_title('Detection')
#     axes[1].imshow(cropped)
#     axes[1].set_title('Segmented')
#     axes[2].imshow(standardized)
#     axes[2].set_title('Standardized')
#     for ax in axes: ax.axis('off')
#     plt.show()
#
#     if result['is_new']:
#         print('\n⚠ POTENTIAL NEW INDIVIDUAL (no confident match)')
#     else:
#         print(f'\n✓ Best match: {result["matches"][0]["individual_id"]}')
#
#     print('\nTop matches:')
#     for m in result['matches']:
#         print(f'  {m["individual_id"]}: {m["score"]:.3f}')

## 4. Register New Individual

In [ ]:
# If the toad is a new individual, register it:

# new_id = input('Enter new individual ID (e.g., TOAD_042): ')
# capture_date = input('Capture date (YYYY-MM-DD): ')
#
# db.add(
#     embedding=result['embedding'],
#     individual_id=new_id,
#     photo_path=filename,
#     capture_date=capture_date,
# )
# db.save(Path('data/models/reid'))
# print(f'Registered {new_id}. Database now has {db.size} embeddings.')

## 5. Export Results

In [ ]:
# Export database summary and results for mark-recapture analysis

# from src.utils.io import export_results_csv
#
# individuals = db.get_individuals()
# print(f'Database contains {len(individuals)} individuals:')
# for ind_id in individuals:
#     count = sum(1 for m in db.metadata if m['individual_id'] == ind_id)
#     print(f'  {ind_id}: {count} sighting(s)')
#
# # Export to CSV
# export_results_csv(db.metadata, Path('results/sightings.csv'))
# files.download('results/sightings.csv')